# Future Crop Challenge — submission: literature CO2 (`24` wheat / `06` maize)

Per-cell yield model fit on **all 39 training years (381–419)** and applied to
the full test window (420–497). Each grid cell gets its own parameter vector,
fit independently — no pooling across locations.

| crop | model | form |
|---|---|---|
| wheat | `24_co2_saturating_multiplier` | `(a + b·tanh(GDD/1000−1) + c·tanh(GDD/1000−1)·tanh(PREC/10) + d·log1p(PRC_MID) − e·tanh(VD/50)) · (1 + γ·h(CO2))`, γ=0.40 **fixed**, `h=(C−400)/(C−400+350)` |
| maize | `06_saturating_vpd` | `a + b·tanh(GDD/1000−1) + c·log1p(PREC) − d·HEAT30/8 − e·tanh(VPD/2.5) + f·log(CO2/380)` (unchanged baseline) |

**Why CO2 on wheat.** Wheat is C3: biophysical crop models of the generating
model's family (CERES, EPIC, APSIM, STICS — all in GGCMI Phase 2) scale
photosynthetically available carbon by a *saturating* multiplier of ambient
CO2. FACE + the Ainsworth & Long optimality model anchor +12–19% at 550 ppm;
this fixed form gives ≈ +12% at 550 ppm and ≈ +27% at the 1108 ppm test
ceiling. **No CO2 parameters are fitted** — γ and the shape constant come from
the literature, so the per-cell fit stays honest (the multiplier is a fixed
per-row vector).

The local 8-year validation window (CO2 ≈ 420–440 ppm) cannot adjudicate this
— the multiplier moves only ±3–4% there — so this notebook exists on mechanism,
not on a held-out win. It passed the pairwise-majority and era-stability gates
(`sandbox/era_stability.md`), which is the most the local data can offer: a
veto, never a confirmation.

**Fit protocol.** Identical to the baseline notebook: closed-form per-cell
ridge on all 39 train years, `θ = (XᵀX + T·ridge·R)⁻¹Xᵀy`, `R = diag(0,1,…,1)`.


In [ ]:
import numpy as np
import pandas as pd
import time

t0 = time.time()

# ---- point DATA_DIR at the local repo for testing ----
DATA_DIR = "/kaggle/input/the-future-crop-challenge/"     # Kaggle
# DATA_DIR = "data/"                                      # local run

RIDGE = 0.1                # ridge weight on params[1:]^2 (intercept exempt)
MIN_TRAIN_YEARS = 3        # per-cell min train rows to fit; fewer -> cell mean


## Data loading

Each feature is one parquet per (variable, crop, split). Row layout is uniform
across files and aligned by the row **index = submission ID**: columns `'0'…'239'`
are the 240-day series. The `soil_co2_*.parquet` files carry the per-row
metadata (`year`, `lon`, `lat`, atmospheric `co2`); `pr` is stored in m/day
and multiplied by 1000 → mm/day to match the sandbox cache pipeline.

Running locally: uncomment the `data/` `DATA_DIR` line. On Kaggle the default
`/kaggle/input/the-future-crop-challenge/` is used.


In [ ]:
def read_meta(crop, split):
    """Metadata carrier parquet: year, lon, lat, atmospheric CO2 per row."""
    df = pd.read_parquet(f"{DATA_DIR}/soil_co2_{crop}_{split}.parquet")
    return df[["year", "lon", "lat", "co2"]]


def read_days(feature, crop, split):
    """Day columns '0'..'239' of one climate feature as float32 (N, 240)."""
    df = pd.read_parquet(f"{DATA_DIR}/{feature}_{crop}_{split}.parquet",
                         columns=[str(i) for i in range(240)])
    return df.to_numpy(dtype=np.float32)


def location_codes(meta_tr, meta_te):
    """Consistent per-cell codes across train and test rows (test subset of train)."""
    both = pd.concat([meta_tr[["lon", "lat"]], meta_te[["lon", "lat"]]],
                     ignore_index=True)
    codes = both.groupby(["lon", "lat"], sort=False).ngroup().to_numpy()
    return codes[: len(meta_tr)], codes[len(meta_tr):]


In [ ]:
GAMMA = {"wheat": 0.40, "maize": 0.03}
CREF, KHALF = 400.0, 350.0


def co2_multiplier(co2, crop):
    """Fixed literature saturating-CO2 multiplier (1 + gamma*h(C)).

    24_co2_saturating_multiplier: yield = weather_base * (1 + gamma*h(C)),
    with h(C) = (C-400)/(C-400+350) and gamma FIXED from literature -
    wheat 0.40 (C3 photosynthetic response; FACE/optimality anchor ~+12% at
    550 ppm), maize 0.03 (C4 near-control). No CO2 parameters are fitted, so
    the per-cell ridge fit stays honest (the multiplier is a fixed per-row
    vector). At the test ceiling (1108 ppm) h~0.67 -> wheat +27%.
    """
    d = co2 - CREF
    h = d / (d + KHALF)
    return 1.0 + GAMMA[crop] * h


def make_design(crop, tasmax, tasmin, pr_mm, co2):
    """Model design matrix; parameters enter linearly -> closed-form ridge.

    wheat -> 24_co2_saturating_multiplier (13 core x fixed CO2 multiplier):
        (a + b*tanh(GDD/1000-1) + c*tanh(GDD/1000-1)*tanh(PREC/10)
           + d*log1p(PRC_MID) - e*tanh(VD/50)) * (1 + gamma*h(CO2))
      -> every design column is its weather feature times the multiplier.
    maize -> 06_saturating_vpd (unchanged benchmark-best):
        a + b*tanh(GDD/1000-1) + c*log1p(PREC) - d*HEAT30/8
          - e*tanh(VPD/2.5) + f*log(CO2/380)
    """
    tmean = 0.5 * (tasmax + tasmin)
    gdd = np.maximum(tmean - 8.0, 0.0).sum(axis=1)
    prec = pr_mm.sum(axis=1)
    prc_mid = pr_mm[:, 80:160].sum(axis=1)
    es = lambda T: 0.6108 * np.exp(17.27 * T / (T + 237.3))   # Magnus, hPa
    vpd = es(tasmax) - es(tasmin)
    heat30 = (tasmax > 30.0).sum(axis=1)
    vpd_mean = vpd.mean(axis=1)
    vd = np.maximum(vpd - 2.0, 0.0).sum(axis=1)
    th = np.tanh(gdd / 1000.0 - 1.0)
    if crop == "wheat":
        m = co2_multiplier(co2, crop)
        return np.column_stack([m, m * th,
                                m * th * np.tanh(prec / 10.0),
                                m * np.log1p(prc_mid), -m * np.tanh(vd / 50.0)])
    return np.column_stack([np.ones_like(th), th, np.log1p(prec),
                            -(heat30 / 8.0), -np.tanh(vpd_mean / 2.5),
                            np.log(np.maximum(co2, 1e-6) / 380.0)])


In [ ]:
def fit_cells(D_tr, y_tr, loc, ridge=RIDGE):
    """Per-cell closed-form ridge: min mean((X@th - y)^2) + ridge*sum(th[1:]^2).

    Exact optimum of the sandbox L-BFGS-B objective (models are linear in the
    parameters once the weather/CO2 features are fixed).
    """
    n_locs = loc.max() + 1
    p = D_tr.shape[1]
    R = np.diag([0.0] + [1.0] * (p - 1))
    params = np.zeros((n_locs, p))
    for c in range(n_locs):
        m = np.where(loc == c)[0]
        yc = y_tr[m]
        if len(m) < MIN_TRAIN_YEARS:
            params[c, 0] = float(yc.mean()) if len(m) else np.nan
            continue
        Xc = D_tr[m]
        try:
            params[c] = np.linalg.solve(Xc.T @ Xc + len(m) * ridge * R,
                                        Xc.T @ yc)
        except np.linalg.LinAlgError:
            params[c, 0] = float(yc.mean())
    return params


def predict_split(crop, tasmax, tasmin, pr_mm, co2, params, loc):
    D = make_design(crop, tasmax, tasmin, pr_mm, co2)
    pred = (D * params[loc]).sum(axis=1)
    return np.maximum(pred, 0.0)


## Fit and predict

For each crop: (1) build a per-row design matrix from the weather (+CO2)
features, (2) fit one ridge-parameter vector per grid cell on train years
381–419, (3) predict every test row (420–497) with its cell's params, floored
at 0.


In [ ]:
preds, crop_stats = {}, {}
for crop in ["maize", "wheat"]:
    print(f"== {crop} ==", flush=True)

    meta_tr = read_meta(crop, "train")
    meta_te = read_meta(crop, "test")
    loc_tr, loc_te = location_codes(meta_tr, meta_te)

    tasmax = read_days("tasmax", crop, "train")
    tasmin = read_days("tasmin", crop, "train")
    pr = read_days("pr", crop, "train") * 1000.0              # m/day -> mm
    y_tr = pd.read_parquet(f"{DATA_DIR}/train_solutions_{crop}.parquet")["yield"].to_numpy()
    D_tr = make_design(crop, tasmax, tasmin, pr, meta_tr["co2"].to_numpy())
    params = fit_cells(D_tr, y_tr, loc_tr)
    del tasmax, tasmin, pr, D_tr

    tasmax = read_days("tasmax", crop, "test")
    tasmin = read_days("tasmin", crop, "test")
    pr = read_days("pr", crop, "test") * 1000.0
    pred = predict_split(crop, tasmax, tasmin, pr,
                         meta_te["co2"].to_numpy(), params, loc_te)
    del tasmax, tasmin, pr

    seen = set(np.unique(loc_tr).tolist())
    unseen = np.array([c not in seen for c in loc_te])
    if unseen.any():
        crop_mean = float(np.mean(y_tr))
        pred[unseen] = crop_mean
        print(f"  WARNING: {unseen.sum()} test rows in cells with no train data", flush=True)

    preds[crop] = pd.Series(pred, index=meta_te.index, name="yield")
    crop_stats[crop] = dict(n_pred=len(pred),
                            pred_mean=float(pred.mean()),
                            pred_std=float(pred.std()),
                            train_mean=float(np.mean(y_tr)),
                            n_floored=int((pred == 0.0).sum()))
    print(f"  rows={len(pred)}  pred_mean={pred.mean():.3f} "
          f"train_mean={np.mean(y_tr):.3f}  floored_at_0={int((pred == 0).sum())}",
          flush=True)


## Assemble submission

Merge predictions (keyed by test ID = parquet index) onto the sample
submission so every row gets a yield. Defensive gap-fill (should not trigger)
uses the overall train mean.


In [ ]:
sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
pred_all = pd.concat([preds["maize"], preds["wheat"]])
sub["yield"] = sub["ID"].map(pred_all)

missing = int(sub["yield"].isna().sum())
if missing:
    sub["yield"] = sub["yield"].fillna(sub["yield"].mean())
    print(f"WARNING: {missing} rows had no prediction, filled with mean", flush=True)

sub.to_csv("submission.csv", index=False)

print("\n= submission =")
print(f"rows: {len(sub)}  NaNs: {missing}  elapsed: {time.time() - t0:.0f}s")
for crop, s in crop_stats.items():
    print(f"{crop:6s} n={s['n_pred']:7d}  pred_mean={s['pred_mean']:.3f}  "
          f"pred_std={s['pred_std']:.3f}  train_mean={s['train_mean']:.3f}  "
          f"floored_at_0={s['n_floored']}")
print(sub.head())


## Notes / caveats

- **Extrapolation**: the test window (420–497) is ~80 years; CO₂ roughly
  doubles (418 → 1108 ppm). This notebook's wheat forecast scales the
  weather core by the fixed saturating multiplier: **+2% at 418 ppm → +27% at
  1108 ppm** (the hinge of the portfolio). Maize is unchanged from benchmark.
- **What the models actually forecast** (mean of row predictions, see the
  run output):
  - **maize** (≈ −19%: same as baseline — heat-driven decline).
  - **wheat** (≈ +19% above train level by test mean): the multiplier mean
    h ≈ 0.46 spanning 0.05→0.67 over the window (1 + 0.4·h̄ ≈ 1.19) pushes the
    per-cell anchor up ~19% while weather terms stay small.
- **Portfolio**: this is the `CO2` arm; `submission_baseline_13_06.ipynb` is
  the `zero-CO2` arm. Test predictions differ ONLY in wheat, by construction.
- **Level anchor / determinism**: same closed-form per-cell ridge; no
  optimizer; reproduce locally with `DATA_DIR = "data/"`.
